# Agent 3 — Program Agent
Matches specific specializations/courses within Agent 2's already-shortlisted universities. Reuses `agent2_final_2tier.csv` (no separate dataset needed — see the framework discussion) since it already carries program-level fields: `curriculum_highlights`, `faculty_research_areas`, `department`, `duration_months`, `stem_designated`.

**What Agent 3 adds on top of Agent 2's output:**
1. Deeper specialization matching — semantic similarity between the student's *stated interests* (not just target_program) and each program's curriculum/faculty research, which Agent 2 doesn't score
2. A parsed `specialization` label pulled out of the program name (e.g. "MS Computer Science - Systems" → `Systems`)
3. `duration` and `mode` fields per the framework's expected output shape
4. A flattened, ranked program list — the exact shape Agent 4 and Agent 6 consume — combining Agent 2's `match_score`/`admit_probability` with this agent's `specialization_fit_score`

**Output contract:** reads `state["matched_universities"]` (Agent 2's university shortlist, with nested `programs`) and `state["profile"]`, writes `state["programs"]` — a flat list, framework shape: `{university, program_name, specialization, duration, mode, match_score}`.

## 1. Setup

### Colab environment fix (run this first, then restart runtime)
Same `PIL._typing._Ink` conflict as Agents 2 and 7 — see those notebooks for details.

**Fix:** run the cell below once, then **Runtime → Restart session**, then re-run from the top.

In [ ]:
!pip install -q -U pillow torchvision transformers sentence-transformers
print("Upgraded. Now go to Runtime -> Restart session, then re-run all cells from the top.")

In [ ]:
!pip install qdrant-client sentence-transformers -q

In [ ]:
import pandas as pd
import numpy as np
import re

from google.colab import files
print("Upload agent2_final_2tier.csv — the same knowledge base Agent 2 uses (no separate dataset needed)")
uploaded = files.upload()

In [ ]:
df = pd.read_csv("agent2_final_2tier.csv")
df = df.fillna("")
print(df.shape)
print(df.columns.tolist())

In [ ]:
from sentence_transformers import SentenceTransformer

# Reuses the same embedding model as Agent 2/7 for consistency (BAAI/bge-small-en-v1.5).
# If Agent 2 is already loaded in this session as `embed_model`, reuse it instead of reloading:
try:
    embed_model
    print("Reusing existing embed_model from an earlier agent in this session.")
except NameError:
    embed_model = SentenceTransformer('BAAI/bge-small-en-v1.5')
    print("Loaded a fresh embed_model.")

## 2. Specialization extraction — parse from program name
Program names in this dataset follow a `"<Degree> <Department> - <Specialization>"` pattern (e.g. `"MS Computer Science - Systems"`). Rule-based, no ML needed — this is a formatting parse, not a prediction.

In [ ]:
def extract_specialization(program_name: str, department: str = "") -> str:
    """Pulls the specialization out of a program name if present, else falls back to department."""
    if " - " in program_name:
        return program_name.split(" - ", 1)[1].strip()
    return department if department else "General"


df["specialization"] = df.apply(
    lambda row: extract_specialization(row["program_name"], row.get("department", "")), axis=1
)
print(df[["program_name", "specialization"]].drop_duplicates().head(10))

## 3. Duration + mode
`duration_months` already exists in the dataset. `mode` (on-campus / online / hybrid) isn't in this CSV — defaults to `"On-campus"` with a flag so you know it's a placeholder, not scraped data. If your team can source this per-program (most course catalogs list it), swap the default for a real lookup.

In [ ]:
DEFAULT_MODE = "On-campus"  # placeholder — this dataset does not carry a delivery-mode column

def get_duration_and_mode(row) -> dict:
    duration_months = row.get("duration_months", None)
    duration_label = f"{int(duration_months)} months" if duration_months not in (None, "", 0) else "Not specified"
    mode = row.get("delivery_mode", DEFAULT_MODE)  # falls back to default if column absent
    return {"duration": duration_label, "mode": mode, "mode_is_placeholder": "delivery_mode" not in row.index}

## 4. Specialization fit scoring — semantic match to student interests
This is what Agent 3 adds that Agent 2 doesn't compute: similarity between the student's *stated interests* (a free-text field, richer than the single `target_program` string Agent 2 uses) and each program's curriculum/faculty research text.

In [ ]:
def compute_specialization_fit(student_interests: str, curriculum_highlights: str, faculty_research_areas: str) -> float:
    """Cosine similarity between student interests and program curriculum/research text."""
    if not student_interests:
        return 0.5  # neutral if the student hasn't provided detailed interests

    program_text = f"{curriculum_highlights} {faculty_research_areas}".strip()
    if not program_text:
        return 0.5

    interest_vec = embed_model.encode(student_interests)
    program_vec = embed_model.encode(program_text)
    cos_sim = float(np.dot(interest_vec, program_vec) / (np.linalg.norm(interest_vec) * np.linalg.norm(program_vec)))
    # cosine similarity is in [-1, 1] — rescale to [0, 1] for consistency with other scores
    return round((cos_sim + 1) / 2, 3)

## 5. Full agent function
Combines Agent 2's `match_score`/`admit_probability` (already computed per program) with this agent's `specialization_fit_score` into a single `combined_score`, then flattens Agent 2's university-grouped output into the framework's expected per-program shape.

In [ ]:
def program_agent(state: dict) -> dict:
    profile = state["profile"]
    student_interests = profile.get("interests_text", "")  # richer free-text field, separate from target_program

    university_matches = state.get("matched_universities", [])
    if not university_matches:
        print("Warning: no matched_universities in state — run Agent 2 first.")
        state["programs"] = []
        return state

    # Build a lookup from (university_name, program_name) -> full row in the knowledge base,
    # so we can pull curriculum/faculty text and duration for each of Agent 2's shortlisted programs.
    df_lookup = df.set_index(["university_name", "program_name"])

    flat_programs = []
    for uni in university_matches:
        for prog in uni["programs"]:
            key = (prog["university_name"], prog["program_name"])
            if key not in df_lookup.index:
                continue  # skip if this exact program row isn't in the knowledge base (shouldn't normally happen)
            row = df_lookup.loc[key]
            if isinstance(row, pd.DataFrame):
                row = row.iloc[0]  # guard against duplicate rows for the same program

            specialization = extract_specialization(prog["program_name"], row.get("department", ""))
            duration_mode = get_duration_and_mode(row)
            fit_score = compute_specialization_fit(
                student_interests, row.get("curriculum_highlights", ""), row.get("faculty_research_areas", "")
            )

            # Combined score: Agent 2's program-level match_score (semantic + tier + gpa + extracurricular)
            # blended with this agent's specialization-specific fit — neither alone is the full picture.
            combined_score = round(0.6 * prog["match_score"] + 0.4 * fit_score, 3)

            flat_programs.append({
                "university": prog["university_name"],
                "program_name": prog["program_name"],
                "specialization": specialization,
                "duration": duration_mode["duration"],
                "mode": duration_mode["mode"],
                "mode_is_placeholder": duration_mode["mode_is_placeholder"],
                "specialization_fit_score": fit_score,
                "match_score": combined_score,
                "admit_probability": prog["admit_probability"],
                "tier": prog["tier"],
            })

    flat_programs = sorted(flat_programs, key=lambda p: p["match_score"], reverse=True)

    state["programs"] = flat_programs
    state["status"] = "program_done"
    return state

## 6. Test run

In [ ]:
MOCK_STATE = {
    "student_id": "test-001",
    "profile": {
        "gpa": 8.6, "gpa_scale": 10.0,
        "target_program": "Computer Science", "target_degree": "MS",
        "test_scores": {"GRE": {"quant": 165}},
        "interests_text": "distributed systems, cloud infrastructure, and large-scale backend engineering",
    },
    "preferences": {"budget_max_usd": 45000, "priority": "research"},
    "extracurricular": {"profile_strength_score": 0.72},
    # Normally this comes from running Agent 2 first — pasted here as a minimal standalone example.
    # If Agent 2 already ran in this session, replace this with its real `result["matched_universities"]`.
    "matched_universities": [
        {
            "university": "Stanford University",
            "programs": [
                {"university_name": "Stanford University", "program_name": "MS Computer Science",
                 "match_score": 0.787, "admit_probability": 0.566, "tier": "Target"},
                {"university_name": "Stanford University", "program_name": "MS Computer Science - Systems",
                 "match_score": 0.793, "admit_probability": 0.566, "tier": "Target"},
            ],
        },
    ],
}

result = program_agent(MOCK_STATE)
for p in result["programs"]:
    print(f"{p['university']} — {p['program_name']}")
    print(f"  specialization={p['specialization']} | duration={p['duration']} | mode={p['mode']}"
          f"{' (placeholder)' if p['mode_is_placeholder'] else ''}")
    print(f"  specialization_fit={p['specialization_fit_score']} | combined match_score={p['match_score']}")
    print()

## 7. Feeding into Agent 4 and Agent 6
`state["programs"]` (flat, ranked list) is what Agent 4 (Financial) scopes cost estimates against, and what Agent 6 (Career) evaluates research/job alignment against — both read this list directly rather than re-deriving it from Agent 2's university-grouped output.

In [ ]:
# Example: how Agent 4 would pick this up
top_program_for_agent4 = result["programs"][0]
print("Top program passed to Agent 4:", top_program_for_agent4["university"], "-", top_program_for_agent4["program_name"])